# 보고동사로 다시 읽기 ① 외부 API 없이 분석하기

맥주 찌꺼기를 다룬 화학 논문 두 편(논문 A: ACS Omega 2025, 논문 B: Molecules 2024)을 **논문 전체**에서 다시 읽습니다. 색칠 → 흐름 그래프 → 클릭 → 원래 문맥으로 돌아가기가 중심입니다.

- 분석 엔진: spaCy 구문분석 + 규칙(`engine_local.py`). **외부 AI API를 부르지 않습니다.** 다만 첫 칸에서 spaCy와 영어 모델을 인터넷으로 내려받습니다.
- 결과 두 층: `core`(입력데이터 명세서 v1.3의 문장별 결과, 스키마 수정 없음)와 `whole`(논문 전체 표현별 결과, 별도 스키마 whole_paper_hits v0.1).
- 결과는 **후보**입니다. 사람이 확인하지 않았고, 초안 주석과 비교한 값은 정확도가 아니라 일치도입니다.
- PDF 추출·DOI 수집은 다루지 않습니다(입력은 이미 전처리된 TXT·TSV). SLA 감성분석과는 다른 분석입니다.

## 1. 설치

In [1]:
print("설치 칸 생략(시험 환경)")

설치 칸 생략(시험 환경)


## 2. 파일 준비
`rv_colab.zip`을 올리면 자동으로 풀립니다. ZIP을 쓸 수 없으면 파일을 하나씩 올려도 됩니다(필수 10개, 선택 2개).

In [2]:
# 파일 준비: rv_colab.zip 하나를 올리거나, 파일을 하나씩 올려도 됩니다.
# ZIP은 rv_colab.zip 하나만 새 임시 폴더에 풀고, 필수 파일이 모두 있을 때만 현재 폴더에 덮어써 옮깁니다
# (이전에 올린 파일이 새 판본과 섞이지 않게). ZIP과 개별 파일을 섞어 올리면 ZIP 내용이 우선합니다.
import os, zipfile, shutil, tempfile, hashlib
NEED = ["rvtool.py", "engine_local.py", "engine_api.py", "viewer_template.html",
        "schemas/core_v1.3.json", "schemas/whole_paper_hits_v0.1.json",
        "inputs/paper_a_input.txt", "inputs/paper_a_structure.tsv", "inputs/paper_b_input.txt", "inputs/paper_b_structure.tsv"]
OPTIONAL = ["annotations/paper_a_draft_v1.3.csv", "annotations/paper_b_draft_v1.3.csv"]
ZIP_NAME = "rv_colab.zip"
def install_zip(zpath):
    tmp = tempfile.mkdtemp(prefix="rv_zip_")
    with zipfile.ZipFile(zpath) as f: f.extractall(tmp)
    base = os.path.join(tmp, "rv_colab") if os.path.isdir(os.path.join(tmp, "rv_colab")) else tmp
    lacking = [rel for rel in NEED if not os.path.exists(os.path.join(base, rel))]
    if lacking:   # 새 ZIP 자체에 필수 파일이 다 있어야 복사한다(이전 파일이 빈자리를 채우지 않게)
        raise FileNotFoundError(f"올린 ZIP에 필수 파일이 없습니다: {lacking}. 아무것도 복사하지 않았습니다. 최신 {ZIP_NAME}을 올려 주세요.")
    for rel in NEED + OPTIONAL:
        src = os.path.join(base, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(rel) or ".", exist_ok=True); shutil.copy2(src, rel)
    print(f"자료 ZIP: {zpath} (SHA-256 {hashlib.sha256(open(zpath, 'rb').read()).hexdigest()[:16]}…)")
def place_loose_files():
    # 하나씩 올린 자료 파일은 이름을 보고 제자리로 옮긴다(같은 이름이 있으면 새 파일로 바꾼다)
    for rel in NEED + OPTIONAL:
        name = os.path.basename(rel)
        if name != rel and os.path.exists(name):
            os.makedirs(os.path.dirname(rel), exist_ok=True); shutil.move(name, rel)
if os.path.exists(ZIP_NAME): install_zip(ZIP_NAME)
place_loose_files()
missing = [p for p in NEED if not os.path.exists(p)]
if missing:
    try:
        from google.colab import files
        print("다음 파일이 없습니다. 올려 주세요(rv_colab.zip 하나 또는 개별 파일):", missing)
        up = list(files.upload())
        zips = [n for n in up if n.endswith(".zip")]
        if len(zips) > 1: raise FileNotFoundError(f"ZIP 파일은 하나만 올려 주세요(올린 ZIP: {zips})")
        if zips: install_zip(zips[0])
        place_loose_files()
    except ImportError:
        pass
missing = [p for p in NEED if not os.path.exists(p)]
print("필수 파일:", "모두 있음" if not missing else f"없음 → {missing}")
print("선택 파일(초안 주석 CSV):", [p for p in OPTIONAL if os.path.exists(p)])
if missing: raise FileNotFoundError(f"필수 파일이 없습니다: {missing}. 올린 뒤 이 칸을 다시 실행하세요.")


필수 파일: 모두 있음
선택 파일(초안 주석 CSV): ['annotations/paper_a_draft_v1.3.csv', 'annotations/paper_b_draft_v1.3.csv']


## 3. 입력 검사
문장을 다시 나누거나 ID를 바꾸지 않습니다. 중복·누락·순서 불일치·빈 문장·문장 안 탭은 오류로 멈춥니다.

In [3]:
import sys, json, importlib
sys.path.insert(0, ".")
import rvtool as rv; importlib.reload(rv)
if not hasattr(rv, "save_validated_run"):
    raise RuntimeError(f"공통 모듈 판본이 이 노트북보다 오래되었습니다({rv.TOOL_VERSION}). 이 노트북과 같은 때 받은 rv_colab.zip을 올려 주세요.")
print("공통 모듈:", rv.TOOL_VERSION)
RUN_ROOT = "rv_runs"            # 실행 결과 저장 폴더(실행마다 새 하위 폴더, 덮어쓰지 않음)
DOC_VERSION = "preproc-v1.2-xml-20261015"   # 입력 판본 이름(입력 파일 머리부에 없어서 여기서 정함)
def load(d):
    ann = f"annotations/{d}_draft_v1.3.csv"
    return rv.load_document(f"inputs/{d}_input.txt", f"inputs/{d}_structure.tsv", d,
                            ann if os.path.exists(ann) else None, document_version=DOC_VERSION)
A, B = load("paper_a"), load("paper_b")
for D in (A, B): print(json.dumps(rv.describe(D), ensure_ascii=False, indent=1))


공통 모듈: rvtool-1.3.1
{
 "document_id": "paper_a",
 "document_version": "preproc-v1.2-xml-20261015",
 "sentences": 249,
 "paragraphs": 64,
 "by_section": {
  "ABS": 10,
  "INT": 24,
  "MET": 67,
  "RD": 139,
  "CON": 9
 },
 "citing_sentences": 49,
 "annotations_loaded": true,
 "hashes": {
  "txt": "9c5be835900932e2fe268efbaa8ba739021250ac60478718eac35370239f9c78",
  "tsv": "dde09f54fbb92d0bfc07384da94f731eba8c67889b4a248dad535e32048f84b9",
  "annotations": "fc53e53b22d18959859248ee4c9339fa5f27e3761616d57a80815e7ddb6fa873"
 }
}
{
 "document_id": "paper_b",
 "document_version": "preproc-v1.2-xml-20261015",
 "sentences": 215,
 "paragraphs": 48,
 "by_section": {
  "ABS": 8,
  "INT": 43,
  "RD": 73,
  "MET": 85,
  "CON": 6
 },
 "citing_sentences": 49,
 "annotations_loaded": true,
 "hashes": {
  "txt": "3a45b159f4e988a85747b9189d42801bbf394b456d966cec86763b0814e035a9",
  "tsv": "6a2bdf82816fc464447eeb4bad522198f54b66e16a789ff66daed574a6fa5848",
  "annotations": "ec5ec4daafb2f9c7edefae6c5f5ba7e

## 4. 소량 시험(정식 저장·다시 읽기·부분 화면까지)
명세서의 연결 시험 문장 두 개(논문 A의 3.2.3-007, 3.3.1-012)만 분석하고, 전체 실행과 똑같이 저장 → 다시 읽기 → 검증 → 부분 화면까지 확인합니다. 기대값은 Claude 판정 초안이며 사람이 확인한 정답이 아닙니다.

In [4]:
import engine_local as L, time; importlib.reload(L)
ids = [i for d, i in rv.SMOKE]
core, whole, excl, f = L.analyze(A, sentence_ids=ids)
rid = rv.new_run_id("local-smoke")
rec = rv.base_record(A, "local", rid, ["core_v1.3", "whole_paper_hits_0.1"], run_kind="smoke", **f)
SMOKE_DIR = rv.save_validated_run(RUN_ROOT, A, rid, rec, core, whole, extra=dict(excluded=excl), expected_ids=ids)   # 검증 실패 시 저장하지 않음
r = rv.load_run(RUN_ROOT, "paper_a", rid)              # 다시 읽기(해시 확인)
for c in r["core"]: print(c)
print("표현 후보:", [(h["sentence_id"], h["text"], h["role"]) for h in r["whole"]])
print("다시 읽은 결과 검사 — core:", rv.validate_core(r["core"], A, ids) or "통과", "/ whole:", rv.validate_whole(r["whole"], A) or "통과")
SMOKE_VIEW = rv.build_viewer([r], [A], f"viewer_local_smoke_{time.strftime('%Y%m%d-%H%M%S')}.html")
print("저장:", SMOKE_DIR, "| 부분 화면:", SMOKE_VIEW, "(분석한 2문장만 색칠, 나머지는 '미분석' 표시)")
try:
    from google.colab import files
    from IPython.display import HTML, display
    display(HTML(open(SMOKE_VIEW, encoding="utf-8").read()))
except ImportError:
    pass

{'document_id': 'paper_a', 'sentence_id': '3.2.3-007', 'has_citation': True, 'citation_numbers': [32], 'reporting_verb': 'reported', 'citation_form': 'integral', 'note': '로컬 규칙 결과(판정 완료 아님)'}
{'document_id': 'paper_a', 'sentence_id': '3.3.1-012', 'has_citation': True, 'citation_numbers': [40, 41], 'reporting_verb': 'reported', 'citation_form': 'non_integral', 'note': '로컬 규칙 결과(판정 완료 아님)'}
표현 후보: [('3.2.3-007', 'reported', 'prior'), ('3.3.1-012', 'reported', 'prior'), ('3.3.1-012', 'added', 'method')]
다시 읽은 결과 검사 — core: 통과 / whole: 통과
저장: rv_runs/paper_a/20261102-202735-local-smoke-eef037 | 부분 화면: viewer_local_smoke_20261102-202736.html (분석한 2문장만 색칠, 나머지는 '미분석' 표시)


## 5. 전체 분석과 저장
기본은 **독립 분석**(초안 주석 CSV를 쓰지 않음)입니다. `USE_ANNOTATIONS=True`는 초안 주석을 그대로 가져와 표시하는 별도 모드이며, 그 결과를 같은 CSV와 비교하면 안 됩니다.

In [5]:
USE_ANNOTATIONS = False
SCOPE = ["core_v1.3", "whole_paper_hits_0.1"]
RUNS = {}
for D in (A, B):
    core, whole, excl, f = L.analyze(D, use_annotations=USE_ANNOTATIONS)
    rid = rv.new_run_id("local")
    rec = rv.base_record(D, "local", rid, SCOPE, **f)
    d = rv.save_validated_run(RUN_ROOT, D, rid, rec, core, whole, extra=dict(excluded=excl))   # 검증 실패 시 저장하지 않음
    RUNS[D.document_id] = rid
    print(D.document_id, "저장:", d, "| 표현", len(whole), "| 제외", len(excl), "| 저장 전 검증: 통과")

paper_a 저장: rv_runs/paper_a/20261102-202736-local-8cad2a | 표현 188 | 제외 87 | 저장 전 검증: 통과


paper_b 저장: rv_runs/paper_b/20261102-202737-local-e0d5b1 | 표현 182 | 제외 105 | 저장 전 검증: 통과


## 6. 검증과 다시 읽기
저장한 파일을 다시 열어 해시가 그대로인지 확인한 뒤 스키마·ID·위치(`sentence[start:end]==text`)를 다시 검사합니다.

In [6]:
for D in (A, B):
    r = rv.load_run(RUN_ROOT, D.document_id, RUNS[D.document_id])
    print(D.document_id, r["record"]["run_id"], "| core:", rv.validate_core(r["core"], D) or "통과", "| whole:", rv.validate_whole(r["whole"], D) or "통과")
print(json.dumps(rv.list_runs(RUN_ROOT), ensure_ascii=False, indent=1)[:1500])

paper_a 20261102-202736-local-8cad2a | core: 통과 | whole: 통과
paper_b 20261102-202737-local-e0d5b1 | core: 통과 | whole: 통과
[
 {
  "document_id": "paper_a",
  "run_id": "20261102-202735-local-smoke-eef037",
  "engine": "local",
  "scope": [
   "core_v1.3",
   "whole_paper_hits_0.1"
  ],
  "run_kind": "smoke",
  "saved_at": "2026-11-02T20:27:36+09:00",
  "status": {
   "requested": 2,
   "succeeded": 2,
   "failed": 0,
   "failed_ids": []
  }
 },
 {
  "document_id": "paper_a",
  "run_id": "20261102-202736-local-8cad2a",
  "engine": "local",
  "scope": [
   "core_v1.3",
   "whole_paper_hits_0.1"
  ],
  "run_kind": "full",
  "saved_at": "2026-11-02T20:27:36+09:00",
  "status": {
   "requested": 249,
   "succeeded": 249,
   "failed": 0,
   "failed_ids": []
  }
 },
 {
  "document_id": "paper_b",
  "run_id": "20261102-202737-local-e0d5b1",
  "engine": "local",
  "scope": [
   "core_v1.3",
   "whole_paper_hits_0.1"
  ],
  "run_kind": "full",
  "saved_at": "2026-11-02T20:27:37+09:00",
  "status": 

## 7. 초안 주석과의 일치도(선택)
초안 주석 CSV가 있을 때만 계산합니다. **Claude 판정 초안과의 일치도이며 정확도가 아닙니다.** 불일치 문장은 어느 쪽이 맞는지 정하지 않은 후보 목록입니다.

In [7]:
for D in (A, B):
    r = rv.load_run(RUN_ROOT, D.document_id, RUNS[D.document_id])
    if r["record"]["use_annotations"]:
        print(D.document_id, "주석 재사용 모드라 일치도를 계산하지 않습니다"); continue
    ag = rv.agreement_with_annotations(r["core"], D, "독립 분석(로컬 규칙)")
    if ag: print(D.document_id, ag["label"], ag["counts"], "불일치 후보", len(ag["disagreement_candidates"]))

paper_a Claude 판정 초안과의 일치도(사람이 확인한 정답이 아니므로 정확도가 아님) {'sentences': 49, 'citation_numbers_same': 49, 'has_verb_same': 41, 'verb_text_same': 35, 'form_same': 49, 'pending_in_draft': 7} 불일치 후보 14
paper_b Claude 판정 초안과의 일치도(사람이 확인한 정답이 아니므로 정확도가 아님) {'sentences': 49, 'citation_numbers_same': 49, 'has_verb_same': 31, 'verb_text_same': 27, 'form_same': 47, 'pending_in_draft': 6} 불일치 후보 22


## 8. 절별 집계
집계 단위는 표현(hit), 분모는 절별 단어 수입니다. 초록은 기본으로 뺍니다. 결과·논의(RD)를 결과와 논의로 나누지 않습니다.

In [8]:
try:
    import pandas as pd
except ImportError:
    pd = None
for D in (A, B):
    r = rv.load_run(RUN_ROOT, D.document_id, RUNS[D.document_id])
    agg = rv.aggregate(D, r["whole"], exclude_abstract=True, failed_ids=r["record"]["processing"]["failed_ids"])
    print(D.document_id, "| 분석 실패 문장:", agg["failed_sentences"])
    if pd is not None: display(pd.DataFrame(agg["rows"]).pivot(index="role", columns="section_type", values="per_1000_words"))

paper_a | 분석 실패 문장: 0


section_type,CON,INT,MET,RD
role,,,,
interpret,6.94,1.76,0.77,2.86
method,27.78,7.04,36.24,12.06
prior,0.00,8.80,6.17,5.08
ref,0.00,0.00,0.77,2.22
result,6.94,0.00,0.77,9.84
uncertain,0.00,1.76,2.31,0.95


paper_b | 분석 실패 문장: 0


section_type,CON,INT,MET,RD
role,,,,
interpret,0.00,0.87,0.00,4.33
method,4.76,12.17,38.10,9.75
prior,0.00,3.48,5.44,3.25
ref,0.00,0.87,0.00,1.63
result,14.29,2.61,0.00,6.50
uncertain,0.00,1.74,1.48,3.79


## 9. 화면 연결: 보고동사로 다시 읽기
저장된 결과로 화면을 만듭니다. 그래프의 점을 누르면 그 구간 문장 → 앞뒤 문맥·문단 → 전체 본문 위치로 갑니다.

In [9]:
import time
out = f"viewer_local_{time.strftime('%Y%m%d-%H%M%S')}.html"
rv.build_viewer([rv.load_run(RUN_ROOT, "paper_a", RUNS["paper_a"]), rv.load_run(RUN_ROOT, "paper_b", RUNS["paper_b"])], [A, B], out)
print("화면 파일:", out)
try:
    from google.colab import files
    from IPython.display import HTML, display
    display(HTML(open(out, encoding="utf-8").read()))
    files.download(out)
except ImportError:
    pass

화면 파일: viewer_local_20261102-202737.html


## 10. 결과 내려받기
`rv_runs` 폴더 전체를 ZIP으로 묶습니다. 이 노트북에서 실행하지 않은 것: 실제 API 분석(② 노트북), 사람 기준의 정확도 평가.

In [10]:
import shutil
z = shutil.make_archive(f"rv_runs_{time.strftime('%Y%m%d-%H%M%S')}", "zip", RUN_ROOT)
print("내려받을 파일:", z)
try:
    from google.colab import files; files.download(z)
except ImportError:
    pass

내려받을 파일: /private/var/folders/6w/dh825y0s1l59l9sysp_1n4h00000gn/T/nbtest_2xj94ctx/rv_runs_20261102-202737.zip
